# Auto-encoding variable bayes

Adapted from https://github.com/pytorch/examples/tree/main/vae

In [ ]:
from __future__ import print_function
import argparse
import pickle

import torch
import torch.utils.data
import torchvision.utils
from torch import nn, optim, Tensor

from torch.nn import functional as F
from torch.optim import Optimizer
from torchvision import datasets, transforms
from torchvision.utils import save_image
import numpy as np

import torchastic
import matplotlib.pyplot as plt

from numbers import Number

from typing import NoReturn, Callable, Sequence, Final, Protocol, TypedDict, Literal, Union, ClassVar, List, FrozenSet, Generic, TypeVar, Set, Tuple, Callable, Iterable, Any, Dict, Iterator,Optional
import dataclasses
from dataclasses import dataclass

from time import perf_counter

import json

%matplotlib notebook


In [ ]:

batch_size: int = 128
"""batch size for training (default: 128)"""

epochs: int = 10
"""number of epochs to train (default: 10)"""
_use_accel: bool = True
"""should accelerator be enabled"""
seed: int = 1
"""rng seed (default: 1)"""
log_interval: int = 10
"""how many batches to wait before logging training status (default: 10)"""

use_accel: bool = _use_accel and torch.accelerator.is_available()

torch.manual_seed(seed)

if use_accel:
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
kwargs = {'num_workers': 1, 'pin_memory': True} if use_accel else {}
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=True, download=True,
                   transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=False, transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=False, **kwargs)

In [ ]:
class VAE(nn.Module):
    """
    An auto-encoding model using a variable bayesian approach.
    """
    def __init__(self, name: str, long_name: str = None):
        super(VAE, self).__init__()
        self.name: str = name
        self.long_name: str = long_name if long_name is not None else name
        self.fc1 = nn.Linear(784, 400)
        self.fc21 = nn.Linear(400, 20)
        self.fc22 = nn.Linear(400, 20)
        self.fc3 = nn.Linear(20, 400)
        self.fc4 = nn.Linear(400, 784)


    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc21(h1), self.fc22(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h3))

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def get_name(self) -> str:
        return self.name

    def get_long_name(self) -> str:
        return self.long_name

    def get_device(self) -> torch.device:
        return next(self.parameters()).device

    def get_dtype(self) -> torch.dtype:
        return next(self.parameters()).dtype


In [ ]:
model_32: VAE = VAE("adam_32", long_name="float32, Adam optimizer").to(device=device, dtype=torch.float32)
optimizer_32: optim.Optimizer = optim.Adam(model_32.parameters(), lr=1e-3)

model_w_32: VAE = VAE("adamW_32", long_name="float32, AdamW optimizer").to(device=device, dtype=torch.float32)
optimizer_w_32: optim.Optimizer = optim.AdamW(model_w_32.parameters(), lr=1e-3)

model_w_16: VAE = VAE("adamW_16", long_name="bfloat16, AdamW optimizer").to(device=device, dtype=torch.bfloat16)
optimizer_w_16: optim.Optimizer = optim.AdamW(model_w_16.parameters(), lr=1e-3)

model_sr_16: VAE = VAE("sr_adamW_16", long_name="bfloat16, Stochastic AdamW optimizer").to(device=device, dtype=torch.bfloat16)
optimizer_sr_16: optim.Optimizer = torchastic.AdamW(model_sr_16.parameters(), lr=1e-3)

In [ ]:
# Reconstruction + KL divergence losses summed over all elements and batch
def loss_function(recon_x, x, mu, logvar,
                  dtype: torch.dtype = torch.get_default_dtype()
    ):
    BCE = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')

    # see Appendix B from VAE paper:
    # Kingma and Welling. Auto-Encoding Variational Bayes. ICLR, 2014
    # https://arxiv.org/abs/1312.6114
    # 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dtype=dtype)

    return torch.sum(BCE + KLD, dtype=dtype) #BCE + KLD


In [ ]:
@dataclass(init=True, frozen=True)
class EpochResult:
    """dataclass for the training results"""
    model_id: str
    epoch: int
    train_loss: Number
    train_duration: float
    test_loss: Number

    comparison_fname: str
    result_fname: str

    comparison_img: Union[None, torch.Tensor]
    result_img: Union[None, torch.Tensor]

    @classmethod
    def make(cls, id: str, epoch: int, train_loss: Number, train_duration: float, test_loss: Number, comp_fname: str, result_fname: str, comp_img: Union[None, torch.Tensor] = None, result_img: Union[None, torch.Tensor] = None) -> "EpochResult":
        return cls(id, epoch, train_loss, train_duration, test_loss, comp_fname, result_fname, comp_img, result_img)

    @classmethod
    def from_json(cls, json_str: str) -> "EpochResult":
        data = json.decoder.JSONDecoder().decode(json_str)
        return cls.make(data["model_id"], data["epoch"], data["train_loss"], data["train_duration"], data["test_loss"], data["comparison_fname"], data["result_fname"])

    def json(self) -> str:
        return json.dumps({"model_id": self.model_id, "epoch": self.epoch, "train_loss": self.train_loss, "train_duration": self.train_duration, "test_loss": self.test_loss, "comparison_fname": self.comparison_fname, "result_fname": self.result_fname}, sort_keys=True, indent=4)

    def __repr__(self) -> str:
        return f"EpochResult(model_id={self.model_id}, epoch={self.epoch}, train_loss={self.train_loss}, train_duration={self.train_duration}, test_loss={self.test_loss}, comparison_fname={self.comparison_fname}, result_fname={self.result_fname})"

dataclass(frozen=True, repr=True)
class ModelResults:
    """dataclass for the overall results of a model"""
    #epochs: tuple[EpochResult, ... ]
    "the per-epoch results of the model"

    #model_file: str

    #model_id: str
    #model_full_name: str

    def __init__(self, all_epochs: Sequence[EpochResult], model_file: str, model_name: str, model_full_name: str):
        self.epochs = tuple(all_epochs)
        self.model_file = model_file
        self.model_id = model_name
        self.model_full_name = model_full_name

    @classmethod
    def make(cls, all_epochs: Sequence[EpochResult], model: VAE, model_filename: str) -> "ModelResults":
        return cls(tuple(e for e in all_epochs), model_filename, model.get_name(), model.get_long_name())

    @property
    def epoch_count(self) -> int:
        return len(self.epochs)


    def __iter__(self) -> Iterator[EpochResult]:
        return iter(self.epochs)

    @property
    def train_durations(self) -> Iterable[float]:
        for e in self.epochs:
            yield e.train_duration
        return StopIteration

    @property
    def train_losses(self) -> Iterable[Number]:
        for e in self.epochs:
            yield e.train_loss
        return StopIteration

    @property
    def test_losses(self) -> Iterable[Number]:
        for e in self.epochs:
            yield e.test_loss
        return StopIteration

    @property
    def comparison_imgs(self) -> Iterable[torch.Tensor]:
        for e in self.epochs:
            yield e.comparison_img
        return StopIteration

    @property
    def result_imgs(self) -> Iterable[torch.Tensor]:
        for e in self.epochs:
            yield e.result_img
        return StopIteration

    @property
    def comparison_fnames(self) -> Iterable[str]:
        for e in self.epochs:
            yield e.comparison_fname
        return StopIteration

    @property
    def result_fnames(self) -> Iterable[str]:
        for e in self.epochs:
            yield e.result_fname
        return StopIteration

    def __len__(self) -> int:
        return len(self.epochs)

    def __getitem__(self, idx: int) -> EpochResult:
        for e in self.epochs:
            if e.epoch == idx:
                return e
        raise IndexError(f"Epoch {idx} does not exist. Consider querying the epochs tuple directly.")

    def json(self) -> str:
        return json.dumps({"model_file": self.model_file, "model_id": self.model_id, "model_full_name": self.model_full_name, "epochs": tuple(e.json() for e in self.epochs)}, sort_keys=True, indent=4)

    @classmethod
    def from_json(cls, json_str: str) -> "ModelResults":
        data = json.decoder.JSONDecoder().decode(json_str)
        return ModelResults(tuple(EpochResult.from_json(e) for e in data["epochs"]), data["model_file"], data["model_id"], data["model_full_name"])


In [ ]:
def train(epoch: int, model: VAE, optimizer: Optimizer) -> tuple[float, Number]:
    train_loss: Number = 0
    start_time: float = perf_counter()
    model.train()
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device=device, dtype=model.get_dtype())
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        loss = loss_function(recon_batch, data, mu, logvar, dtype=model.get_dtype())
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader),
                loss.item() / len(data)))
    end_time: float = perf_counter()
    duration: float = end_time - start_time
    avg_loss: Number = train_loss / len(train_loader.dataset)
    print('====> Epoch: {} Average loss: {:.4f}'.format(
          epoch, avg_loss))
    return duration, avg_loss


In [ ]:

def test(epoch, model: VAE, optimizer: Optimizer) -> tuple[Number, str, torch.Tensor, str] :
    model.eval()
    test_loss: Number = 0
    comp_fname: str = 'results/img/reconstruction_' + str(epoch) + '_' + str(model.get_name()) +  '.png'
    comp_img: torch.Tensor = torch.empty(0)
    comp_epoch: str = ""
    with torch.no_grad():
        for i, (data, _) in enumerate(test_loader):
            data = data.to(device).to(device=device, dtype=model.get_dtype())
            recon_batch, mu, logvar = model(data)
            test_loss += loss_function(recon_batch, data, mu, logvar).item()
            if i == 0:
                n = min(data.size(0), 8)
                #comp_img = 'results/reconstruction_' + str(epoch) + '_' + str(model.get_name()) +  '.png'
                comparison = torchvision.utils.make_grid(
                    torch.cat([data[:n],
                         recon_batch.view(batch_size, 1, 28, 28)[:n]]
                    )
                )
                #comparison = torch.cat([data[:n],
                #                      recon_batch.view(batch_size, 1, 28, 28)[:n]])
                #comp_img = 'results/reconstruction_' + str(epoch) + '_' + str(model.get_name()) +  '.png'
                save_image(comparison.cpu(),
                         comp_fname, nrow=n)
                #comp_img = comparison.cpu().permute(0,2,3,1).reshape(2 * 28, 8 * 28, 1)
                comp_epoch = str(epoch)

    test_loss /= len(test_loader.dataset)
    print('====> Test set loss: {:.4f}  {:s}'.format(test_loss, model.get_name()))
    return test_loss, comp_fname, comp_img, comp_epoch

In [ ]:

def test_and_train(model: VAE, optimizer: Optimizer) -> ModelResults:
    images: list[torch.Tensor] = []
    captions: list[str] = []

    epoch_results: list[EpochResult] = []

    plt.figure(figsize=(10, 4))
    plt.title(f"per-epoch results for model {str(model.get_name())}")
    plt.xlabel("epoch")
    plt.ylabel("loss")

    plt.xticks(np.arange(epochs), np.arange(1, epochs+1))

    #fig_scale : float = 2
    #plt.figure(figsize=(0.5+(2*fig_scale),0.25+(epochs * fig_scale)))
    #plt.title(f"Epoch outputs for model {str(model.get_name())}")

    for epoch in range(1, epochs + 1):
        train_duration, train_loss = train(epoch, model, optimizer)
        test_loss, comp_fname, comp_img, comp_epoch = test(epoch, model, optimizer)

        res_img: torch.Tensor
        res_fname: str = 'results/img/sample_' + str(epoch) + '_' + str(model.get_name()) + '.png'
        if comp_img is not None:
            pass
            #plt.subplot(epochs, 2 ,(2*epoch)-1)
            #plt.imshow(comp_img.to(device=device, dtype=torch.get_default_dtype()), cmap=plt.get_cmap('gray'))
            #plt.xticks([])
            #plt.yticks([])
            #plt.grid(False)
            #plt.xlabel("comparison, epoch {:s}, loss {:.4f}".format(comp_epoch, loss))


        with torch.no_grad():
            sample = torch.randn(64, 20).to(device=device, dtype=model.get_dtype())
            sample = model.decode(sample).cpu()
            #res_fname = 'results/sample_' + str(epoch) + '_' + str(model.get_name()) + '.png'

            #save_image(sample.view(64, 1, 28, 28),
            #           'results/sample_' + str(epoch) + '_' + str(model.get_name()) + '.png')
            #images.append(sample.view(64,1,28,28))
            res_img = torchvision.utils.make_grid(sample.view(64,1,28,28), nrow=8)
            save_image(res_img, res_fname)
            captions.append(f"epoch {str(epoch)}")

            #plt.subplot(epochs,2,2*epoch)
            #plt.imshow(sample.view(64, 1, 28, 28).permute(0,2,3,1).reshape(28 * 8, 28 * 8, 1).to(device=device, dtype=torch.get_default_dtype()), cmap=plt.get_cmap('gray'))
            #plt.xticks([])
            #plt.yticks([])
            #plt.grid(False)
            #plt.xlabel("epoch {:f}, test set loss {:.4f}".format(epoch, loss))
        epoch_results.append(EpochResult.make(model.get_name(), epoch, train_loss, train_duration, test_loss, comp_fname=comp_fname, result_fname=res_fname, comp_img=comp_img, result_img=res_img))

    model_fname: str = 'models/' + str(model.get_name()) + '.pt'
    torch.save(model.state_dict(), model_fname)

    results = ModelResults.make(epoch_results, model, model_fname)

    plt.plot(results.train_losses, label="train loss")
    plt.plot(results.test_losses, label="test loss")

    plt.legend()
    plt.show()

    return results
    #plt.show()



In [ ]:
results_adam_32: ModelResults = test_and_train(model_32, optimizer_32)

json.dump(results_adam_32.json(), open("results/adam_32.json", "w"))
pickle.dump(results_adam_32, open("results/adam_32.pkl", "wb"))


In [ ]:
results_adamw_32: ModelResults = test_and_train(model_w_32, optimizer_w_32)

json.dump(results_adamw_32.json(), open("results/adamw_32.json", "w"))
pickle.dump(results_adamw_32, open("results/adamw_32.pkl", "wb"))

In [ ]:
results_adamw_16: ModelResults = test_and_train(model_w_16, optimizer_w_16)

json.dump(results_adamw_16.json(), open("results/adamw_16.json", "w"))
pickle.dump(results_adamw_16, open("results/adamw_16.pkl", "wb"))

In [ ]:
results_sr_16: ModelResults = test_and_train(model_sr_16, optimizer_sr_16)

json.dump(results_sr_16.json(), open("results/sr_adamw_16.json", "w"))
pickle.dump(results_sr_16, open("results/adamw_16.pkl", "wb"))